In [1]:
LEAGUE = "brasileirao-serie-a"
SEASON = "2024"
ENVIRONMENT = "prd"
REPROC_MODE = True

In [2]:
import boto3
import json
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("raw") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://cvmc-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

s3 = boto3.client(
    's3',
    endpoint_url='http://cvmc-minio:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin'
)

print(f"Spark version: {spark.version}")
print("Delta + MinIO + S3 configurados!")

Spark version: 3.5.0
Delta + MinIO + S3 configurados!


In [3]:
# =====================
# PARÂMETROS DE EXECUÇÃO
# =====================
# LEAGUE = "brasileirao-serie-a"
# SEASON = "2026"
# ENVIRONMENT = "dev"
# REPROC_MODE = True

# =====================
# CONFIGURAÇÕES FIXAS
# =====================
BUCKET = "eng-prd-data-lake"

LANDING_PATH = f"s3a://{BUCKET}/{ENVIRONMENT}/landing/{LEAGUE}"
RAW_PATH = f"s3a://{BUCKET}/{ENVIRONMENT}/raw/{LEAGUE}"

print(f"League: {LEAGUE} | Season: {SEASON} | Env: {ENVIRONMENT} | Reproc: {REPROC_MODE}")
print(f"Landing: {LANDING_PATH}")
print(f"Raw:     {RAW_PATH}")

League: brasileirao-serie-a | Season: 2024 | Env: prd | Reproc: True
Landing: s3a://eng-prd-data-lake/prd/landing/brasileirao-serie-a
Raw:     s3a://eng-prd-data-lake/prd/raw/brasileirao-serie-a


In [4]:
from pyspark.sql.functions import current_timestamp, lit, explode, col
from pyspark.sql.types import StructType, ArrayType

# Habilita a inferência automática de schema para o Streaming conseguir ler a Landing
spark.conf.set("spark.sql.streaming.schemaInference", "true")

# Função que vai tratar e salvar cada lote de dados novos
def process_micro_batch(df_micro_batch, batch_id, raw_target_path):
    print(f"  -> Processando lote {batch_id}...")
    
    # Se o lote vier vazio, ignora e segue o jogo
    if df_micro_batch.isEmpty():
        print("  -> Lote vazio, pulando...")
        return
        
    # 1. Tratamento básico (Explode e novas colunas)
    df = df_micro_batch.withColumn("record", explode(col("data"))) \
                       .drop("data") \
                       .withColumn("ingested_at", current_timestamp()) \
                       .withColumn("league", lit(LEAGUE)) \
                       .withColumn("season", lit(SEASON))

    # 2. Lógica de Proteção de Schema (Nosso Auto Loader "Rescue" caseiro!)
    try:
        # Tenta ler o Delta existente
        df_existing = spark.read.format("delta").load(raw_target_path)
        existing_schema = df_existing.schema
        existing_root_fields = {f.name: f.dataType for f in existing_schema.fields}
        
        # 2.1 Trata raiz
        for f in df.schema.fields:
            if f.name in existing_root_fields and f.name != "record":
                ext_type = existing_root_fields[f.name]
                if type(f.dataType) != type(ext_type):
                    df = df.withColumn(f.name, col(f.name).cast(ext_type))
        
        # 2.2 Trata struct record
        if "record" in existing_root_fields:
            ext_record_schema = existing_root_fields["record"]
            new_record_schema = df.select("record").schema[0].dataType
            
            if isinstance(ext_record_schema, StructType) and isinstance(new_record_schema, StructType):
                existing_rec_fields = {f.name: f.dataType for f in ext_record_schema.fields}
                for f in new_record_schema.fields:
                    if f.name in existing_rec_fields:
                        ext_type = existing_rec_fields[f.name]
                        new_type = f.dataType
                        
                        if type(ext_type) != type(new_type):
                            print(f"  [WARN] Conflito em {f.name}. Resolvendo...")
                            # Se for tipo primitivo, forçamos o cast
                            if not isinstance(ext_type, (StructType, ArrayType)) and not isinstance(new_type, (StructType, ArrayType)):
                                df = df.withColumn("record", col("record").withField(f.name, col(f"record.{f.name}").cast(ext_type)))
                            # Se a estrutura for incompatível, simulamos o "rescue" renomeando a coluna
                            else:
                                new_col_name = f"{f.name}_rescued"
                                df = df.withColumn("record", col("record").withField(new_col_name, col(f"record.{f.name}"))) \
                                       .withColumn("record", col("record").dropFields(f.name))
                                print(f"  -> Dado salvo em {new_col_name}")
    except Exception as e:
        # Se der erro (ex: tabela não existe ainda), o jogo segue para criar a primeira versão
        pass

    # 3. Grava o lote limpo e protegido no Delta Lake
    df.write.format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .partitionBy("season") \
        .save(raw_target_path)
    
    print(f"  -> Lote {batch_id} gravado com sucesso!")

def ingest_to_raw(data_type):
    # O path agora aponta direto pra pasta, e não pra arquivos individuais
    landing_path = f"s3a://{BUCKET}/{ENVIRONMENT}/landing/{LEAGUE}/season={SEASON}/{data_type}/"
    raw_path = f"{RAW_PATH}/{data_type}"
    
    # É aqui que a mágica acontece: o diretório de checkpoint do Spark Streaming
    checkpoint_path = f"s3a://{BUCKET}/{ENVIRONMENT}/checkpoints/raw/{LEAGUE}/{SEASON}/{data_type}"

    print(f"[INFO] {data_type} — Iniciando captura em streaming...")

    # Lê os JSONs usando readStream
    df_stream = spark.readStream \
        .format("json") \
        .option("multiline", "true") \
        .load(landing_path)

    # Inicia a gravação passando os dados para o foreachBatch
    query = df_stream.writeStream \
        .foreachBatch(lambda df, epoch_id: process_micro_batch(df, epoch_id, raw_path)) \
        .option("checkpointLocation", checkpoint_path) \
        .trigger(availableNow=True) \
        .start()

    # Aguarda o lote terminar de processar
    query.awaitTermination()
    print(f"[OK] {data_type} finalizado!\n")

for data_type in ["standing", "rounds", "events", "players", "team_stats", "player_stats"]:
    ingest_to_raw(data_type)

[INFO] standing — Iniciando captura em streaming...
[OK] standing finalizado!

[INFO] rounds — Iniciando captura em streaming...
[OK] rounds finalizado!

[INFO] events — Iniciando captura em streaming...
  -> Processando lote 0...


StreamingQueryException: [STREAM_FAILED] Query [id = 002ce9a8-ba58-40a5-a198-16e3f7ad448b, runId = 3b705f8e-0584-4836-ae19-22134fcb2589] terminated with exception: An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/pyspark/sql/utils.py", line 120, in call
    raise e
  File "/usr/local/spark/python/pyspark/sql/utils.py", line 117, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
  File "/tmp/ipykernel_22199/515333639.py", line 91, in <lambda>
    .foreachBatch(lambda df, epoch_id: process_micro_batch(df, epoch_id, raw_path)) \
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_22199/515333639.py", line 69, in

In [ ]:
# def clean_s3_prefix(prefix):
#     paginator = s3.get_paginator('list_objects_v2')
#     pages = paginator.paginate(Bucket=BUCKET, Prefix=prefix)
#     for page in pages:
#         if 'Contents' in page:
#             for obj in page['Contents']:
#                 s3.delete_object(Bucket=BUCKET, Key=obj['Key'])
#     print(f"Limpeza de ficheiros concluída para: {prefix}")

# # Apaga as tabelas Delta da Raw e os Checkpoints do Streaming
# clean_s3_prefix(f"{ENVIRONMENT}/raw/{LEAGUE}/")
# clean_s3_prefix(f"{ENVIRONMENT}/checkpoints/raw/{LEAGUE}/")